# Export to DICOM using module

This notebook keeps the **query logic in notebook** and delegates export logic to `eyened_orm.export.dicom_export`.


In [ ]:
import os
from pathlib import Path

import numpy as np

from sqlalchemy import select

from eyened_orm import Creator, Database, ImageInstance, Patient, Project, Segmentation, Series, Study
from eyened_orm.export import DicomExportConfig, export_instances_to_dicom

# --- selection/query config ---
PROJECT_NAME = "PROJECT_NAME"
PATIENT_IDENTIFIER = None  # e.g. "P1234"
MAX_INSTANCES = 20 # Integer or None to export all
INCLUDE_INACTIVE = False
MAX_IMAGES_PER_PATIENT = 5  # e.g. 10 to cap images per patient

# --- export config ---
OUTPUT_DIR = Path("/path/to/output/dir/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PSEUDONYMIZE_PATIENT_IDS = True
PSEUDONYM_SALT = os.environ["DICOM_EXPORT_PSEUDONYM_SALT"]  
PSEUDONYM_PREFIX = "ANONYM_EMC_"

OFFSET_DATES_PER_PATIENT = True
DATE_OFFSET_SALT = os.environ["DICOM_EXPORT_DATE_OFFSET_SALT"]  
DATE_OFFSET_MIN_DAYS = -365*3
DATE_OFFSET_MAX_DAYS = 365*3

KEYFILE_PATH =  Path("/path/to/keyfile.csv") ## Path to keyfile with patient and date information
IMAGE_KEYFILE_PATH =  Path("/path/to/image_keyfile.csv") ## Path to keyfile with image information


# --- segmentation export config ---
SEGMENTATION_CREATOR_NAME = "CreatorName"
SEGMENTATION_OUTPUT_DIR = OUTPUT_DIR / "segmentations_npz"
SEGMENTATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

db = Database()





In [ ]:
def query_instances(session):
    # Keep query drafting in notebook so users can customize freely.
    stmt = (
        select(ImageInstance)
        .join(Series)
        .join(Study)
        .join(Patient)
        .join(Project)
        # .where(Patient.PatientIdentifier.in_(["EXAMPLE-0001", "EXAMPLE-0002"]))
        .where(Project.ProjectName == PROJECT_NAME)
        .order_by(Patient.PatientIdentifier.asc(), ImageInstance.ImageInstanceID.asc())
    )

    if not INCLUDE_INACTIVE:
        stmt = stmt.where(~ImageInstance.Inactive)
    if PATIENT_IDENTIFIER:
        stmt = stmt.where(Patient.PatientIdentifier == PATIENT_IDENTIFIER)

    instances = session.scalars(stmt).all()

    if MAX_IMAGES_PER_PATIENT is not None:
        by_patient_count: dict[str, int] = {}
        filtered: list[ImageInstance] = []
        for im in instances:
            pid = im.Patient.PatientIdentifier or "UNKNOWN"
            current = by_patient_count.get(pid, 0)
            if current >= MAX_IMAGES_PER_PATIENT:
                continue
            filtered.append(im)
            by_patient_count[pid] = current + 1
        instances = filtered

    if MAX_INSTANCES is not None:
        instances = instances[:MAX_INSTANCES]

    return instances


with db.get_session() as session:
    instances = query_instances(session)
    print(len(instances))

print(f"Selected {len(instances)} instance(s)")
# for im in instances[:100]:
#     print(f"- id={im.ImageInstanceID} public_id={im.PublicID} series={im.SeriesID} modality={im.Modality}")


In [ ]:
with db.get_session() as session:
    instances = query_instances(session)

    print(f"Selected {len(instances)} instance(s)")
    for im in instances:
        print(
            f"- id={im.ImageInstanceID} public_id={im.PublicID} "
            f"series={im.SeriesID} modality={im.Modality}"
        )

    config = DicomExportConfig(
        output_dir=OUTPUT_DIR,
        pseudonymize_patient_ids=PSEUDONYMIZE_PATIENT_IDS,
        pseudonym_salt=PSEUDONYM_SALT,
        pseudonym_prefix=PSEUDONYM_PREFIX,
        offset_dates_per_patient=OFFSET_DATES_PER_PATIENT,
        date_offset_salt=DATE_OFFSET_SALT,
        date_offset_min_days=DATE_OFFSET_MIN_DAYS,
        date_offset_max_days=DATE_OFFSET_MAX_DAYS,
        keyfile_path=KEYFILE_PATH,
        image_keyfile_path=IMAGE_KEYFILE_PATH,
        include_inactive=INCLUDE_INACTIVE,
    )

    result = export_instances_to_dicom(
        session=session,
        instances=instances,
        config=config,
    )

all_exported_paths = list(result.exported_paths)
all_image_keyfiles = [Path(result.image_keyfile_path)] if result.image_keyfile_path else []

print(f"Requested: {result.requested_count}")
print(f"Exported: {result.exported_count}")
if result.keyfile_path:
    print(f"Patient/date keyfile: {result.keyfile_path}")
if result.image_keyfile_path:
    print(f"Image filename keyfile: {result.image_keyfile_path}")

for p in all_exported_paths[:10]:
    print(f"- {p.name}")



In [ ]:
# Export segmentations by creator to .npz (one file per image)
# Keys in each .npz are feature names.
# Uses generated image filename keyfile to match DICOM filenames.

import csv


# Build mapping from Image PublicID -> exported DICOM stem.
public_id_to_stem: dict[str, str] = {}
if all_image_keyfiles:
    for image_keyfile in all_image_keyfiles:
        if not image_keyfile.exists():
            continue
        with image_keyfile.open("r", newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                exported_filename = row.get("exported_filename", "")
                public_id = row.get("image_public_id", "")
                if exported_filename and public_id:
                    public_id_to_stem[public_id] = Path(exported_filename).stem
else:
    print("Warning: image filename keyfile(s) not found; falling back to PublicID-based naming.")

with db.get_session() as session:
    image_ids = [im.ImageInstanceID for im in instances]

    if not image_ids:
        print("No selected instances; skipping segmentation export.")
    else:
        segmentations = (
            session.query(Segmentation)
            .join(Creator)
            .filter(Segmentation.ImageInstanceID.in_(image_ids))
            .filter(Segmentation.Inactive == False)
            .filter(Creator.CreatorName == SEGMENTATION_CREATOR_NAME)
            .all()
        )

        by_image: dict[int, list[Segmentation]] = {}
        for seg in segmentations:
            if seg.ImageInstanceID is None:
                continue
            by_image.setdefault(seg.ImageInstanceID, []).append(seg)

        exported_npz = []
        for im in instances:
            segs = by_image.get(im.ImageInstanceID, [])
            if not segs:
                continue

            payload: dict[str, np.ndarray] = {}
            feature_seen: dict[str, int] = {}

            for seg in segs:
                data = seg.read_data()
                if data is None:
                    continue

                arr = np.asarray(data)

                base_key = seg.Feature.FeatureName
                idx = feature_seen.get(base_key, 0) + 1
                feature_seen[base_key] = idx
                key = base_key if idx == 1 else f"{base_key}__{idx}"
                payload[key] = arr

            if not payload:
                continue

            dicom_stem = public_id_to_stem.get(im.PublicID, im.PublicID)
            out_path = SEGMENTATION_OUTPUT_DIR / f"{dicom_stem}_segmentations.npz"
            np.savez_compressed(out_path, **payload)
            exported_npz.append(out_path)

        print(
            f"Exported {len(exported_npz)} segmentation file(s) for creator '{SEGMENTATION_CREATOR_NAME}' "
            f"to {SEGMENTATION_OUTPUT_DIR.resolve()}"
        )
        for p in exported_npz[:10]:
            print(f"- {p.name}")


In [ ]:
# Optional: inspect first exported DICOM
import pydicom

if all_exported_paths:
    ds_check = pydicom.dcmread(str(all_exported_paths[0]))
    print(ds_check)
